# Synthetic Control File for Employee Records
- Goal: Generate synthetic employee records for testing and development purposes.

## Imports

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import date, datetime
from abc import ABC, abstractmethod
from typing import Dict, List, Optional, Literal, Any

import numpy as np
import pandas as pd

# Synthetic data
from faker import Faker

## Configs
- This configuration will drive downstream generators (employees -> payroll -> GL).
- Keep config objects typed and deterministic (seeded) for reproducibility.

In [2]:
LocationStrategy = Literal["NY", "SF", "REMOTE"]
JobFamily = Literal["ENGINEERING", "SALES", "PRODUCT", "G&A"]
EmailUniquenessStrategy = Literal["append_id", "dedupe_counter"]

In [3]:
@dataclass(frozen=True)
class CompanyConfig:
    company_name: str = "Gamma Software Solutions, Inc."
    company_domain: str = "gammasoftware.com"
    entity_id: str = "ENT100"
    currency: str = "USD"


@dataclass(frozen=True)
class TimeConfig:
    start_date: date = date(2020, 1, 1)
    as_of_date: date = date(2026, 1, 1)


@dataclass(frozen=True)
class PopulationConfig:
    start_headcount: int = 3
    target_headcount: int = 105


@dataclass(frozen=True)
class IdConfig:
    employee_id_prefix: str = "EMP"
    employee_id_width: int = 6


@dataclass(frozen=True)
class PolicyConfig:
    include_terminations: bool = False
    termination_rate_annual: float = 0.0
    email_uniqueness_strategy: EmailUniquenessStrategy = "append_id"


@dataclass(frozen=True)
class DistributionConfig:
    # high-level organizational job family mix
    job_family_weights: Dict[JobFamily, float] = field(
        default_factory=lambda: {
            "ENGINEERING": 0.5,
            "SALES": 0.15,
            "PRODUCT": 0.15,
            "G&A": 0.2,
        }
    )

    # location mix
    location_weights: Dict[LocationStrategy, float] = field(
        default_factory=lambda: {"NY": 0.4, "SF": 0.3, "REMOTE": 0.3}
    )

    # job family levels (L1 to L6)
    level_weights_by_job_family: Dict[JobFamily, Dict[str, float]] = field(
        default_factory=lambda: {
            "ENGINEERING": {
                "L1": 0.10,
                "L2": 0.20,
                "L3": 0.25,
                "L4": 0.25,
                "L5": 0.15,
                "L6": 0.05,
            },
            "SALES": {
                "L1": 0.15,
                "L2": 0.25,
                "L3": 0.25,
                "L4": 0.20,
                "L5": 0.10,
                "L6": 0.05,
            },
            "PRODUCT": {
                "L1": 0.10,
                "L2": 0.25,
                "L3": 0.25,
                "L4": 0.20,
                "L5": 0.15,
                "L6": 0.05,
            },
            "G&A": {
                "L1": 0.20,
                "L2": 0.30,
                "L3": 0.25,
                "L4": 0.15,
                "L5": 0.08,
                "L6": 0.02,
            },
        }
    )


@dataclass(frozen=True)
class RunConfig:
    seed: int = 42
    source: str = "SYNTHETIC_V1"
    created_at: datetime = field(default_factory=lambda: datetime.now())


@dataclass(frozen=True)
class GlobalConfig:
    company: CompanyConfig = field(default_factory=CompanyConfig)
    time: TimeConfig = field(default_factory=TimeConfig)
    population: PopulationConfig = field(default_factory=PopulationConfig)
    ids: IdConfig = field(default_factory=IdConfig)
    policy: PolicyConfig = field(default_factory=PolicyConfig)
    dist: DistributionConfig = field(default_factory=DistributionConfig)
    run: RunConfig = field(default_factory=RunConfig)


CONFIG = GlobalConfig()
CONFIG

GlobalConfig(company=CompanyConfig(company_name='Gamma Software Solutions, Inc.', company_domain='gammasoftware.com', entity_id='ENT100', currency='USD'), time=TimeConfig(start_date=datetime.date(2020, 1, 1), as_of_date=datetime.date(2026, 1, 1)), population=PopulationConfig(start_headcount=3, target_headcount=105), ids=IdConfig(employee_id_prefix='EMP', employee_id_width=6), policy=PolicyConfig(include_terminations=False, termination_rate_annual=0.0, email_uniqueness_strategy='append_id'), dist=DistributionConfig(job_family_weights={'ENGINEERING': 0.5, 'SALES': 0.15, 'PRODUCT': 0.15, 'G&A': 0.2}, location_weights={'NY': 0.4, 'SF': 0.3, 'REMOTE': 0.3}, level_weights_by_job_family={'ENGINEERING': {'L1': 0.1, 'L2': 0.2, 'L3': 0.25, 'L4': 0.25, 'L5': 0.15, 'L6': 0.05}, 'SALES': {'L1': 0.15, 'L2': 0.25, 'L3': 0.25, 'L4': 0.2, 'L5': 0.1, 'L6': 0.05}, 'PRODUCT': {'L1': 0.1, 'L2': 0.25, 'L3': 0.25, 'L4': 0.2, 'L5': 0.15, 'L6': 0.05}, 'G&A': {'L1': 0.2, 'L2': 0.3, 'L3': 0.25, 'L4': 0.15, 'L5':

In [4]:
# - Reproducibility helper
def _rng(seed: int) -> np.random.Generator:
    """
    Return a numpy Generator seeded for reproducible random draws.

    Parameters
    ----------
    seed : int
        Integer seed to initialize the RNG.

    Returns
    -------
    numpy.random.Generator
        A Generator instance initialized with `seed`.
    """
    return np.random.default_rng(seed)


# - Convert distribution configs into vector of information
def normalize_weights(weight_by_key: Dict[str, float]) -> Dict[str, float]:
    """Return weights normalized to sum of 1"""
    total_weight = float(sum(weight_by_key.values()))
    if total_weight < 0:
        raise ValueError("Weights must sum to a positive value")
    return {key: value / total_weight for key, value in weight_by_key.items()}


def sample_weighted(
    rng: np.random.Generator, weight_by_key: Dict[str, float], sample_size: int
) -> List[str]:
    """Sample config information relative to defined weights"""
    normalized_weights = normalize_weights(weight_by_key)
    categories = list(normalized_weights.keys())
    probabilities = np.array([normalized_weights[k] for k in categories], dtype=float)

    # Generate samples
    samples = rng.choice(categories, size=sample_size, replace=True, p=probabilities)

    return samples.tolist()


# Validate
rng = _rng(seed=CONFIG.run.seed)
sample_weighted(rng, CONFIG.dist.location_weights, 10)

['REMOTE',
 'SF',
 'REMOTE',
 'SF',
 'NY',
 'REMOTE',
 'REMOTE',
 'REMOTE',
 'NY',
 'SF']

## Schema Definition
- Define the schema for employee records including fields such as employee ID, name, department, role, hire date, and salary
- In productionizing, use Pydantic

In [5]:
# specification of each column in the table schema
@dataclass(frozen=True)
class ColumnSpec:
    name: str
    dtype: str
    nullable: bool = True
    unique: bool = False
    description: str = ""


@dataclass(frozen=True)
class TableSchema:
    name: str
    columns: List[ColumnSpec]
    primary_key: Optional[str] = None

    def column_names(self) -> List[str]:
        return [c.name for c in self.columns]

    def dtype_map(self) -> Dict[str, str]:
        return {c.name: c.dtype for c in self.columns}

    def unique_columns(self) -> List[str]:
        return [c.name for c in self.columns if c.unique]

    def required_columns(self) -> List[str]:
        return [c.name for c in self.columns if not c.nullable]


# Employee table dimensions
DIM_EMPLOYEE_SCHEMA = TableSchema(
    name="dim_employee",
    primary_key="employee_id",
    columns=[
        # employee identity
        ColumnSpec(
            name="employee_id",
            dtype="string",
            nullable=False,
            unique=True,
            description="Employee ID (e.g. E1000000)",
        ),
        ColumnSpec(
            name="first_name",
            dtype="string",
            nullable=False,
            description="Employee first name",
        ),
        ColumnSpec(
            name="last_name",
            dtype="string",
            nullable=False,
            description="Employee last name",
        ),
        ColumnSpec(
            name="full_name",
            dtype="string",
            nullable=False,
            description="Employee full name",
        ),
        ColumnSpec(
            name="email",
            dtype="string",
            nullable=False,
            unique=True,
            description="Corporate email",
        ),
        ColumnSpec(
            name="phone", dtype="string", nullable=True, description="Phone number"
        ),
        # employment metadata
        ColumnSpec(
            name="hire_date",
            dtype="datetime64[ns]",
            nullable=False,
            description="Hiring date",
        ),
        ColumnSpec(
            name="terminatation_date",
            dtype="datetime64[ns]",
            nullable=False,
            description="Termination date (NaT if active)",
        ),
        ColumnSpec(
            name="employment_status",
            dtype="string",
            nullable=False,
            description="Active or Terminated",
        ),
        ColumnSpec(
            name="job_family",
            dtype="string",
            nullable=False,
            description="ENGINEERING, SALES, PRODUCT, or G&A",
        ),
        ColumnSpec(
            name="level", dtype="string", nullable=False, description="L1, L2, ..., L6"
        ),
        ColumnSpec(
            name="title",
            dtype="string",
            nullable=True,
            description="Optional title based on job family and level",
        ),
        # financial / organizational
        ColumnSpec(
            name="department",
            dtype="string",
            nullable=False,
            description="Job family unless sub-departments are needed",
        ),
        ColumnSpec(
            name="cost_center",
            dtype="string",
            nullable=False,
            description="Cost center code",
        ),
        ColumnSpec(
            name="manager_id",
            dtype="string",
            nullable=True,
            description="Manager of employee; CEO excluded",
        ),
        ColumnSpec(
            name="location",
            dtype="string",
            nullable=False,
            description="NY, SF, REMOTE",
        ),
        ColumnSpec(
            name="state",
            dtype="string",
            nullable=False,
            description="State/province proxy for tax treatment",
        ),
        # other metadata
        ColumnSpec(
            "created_at",
            "datetime64[ns]",
            nullable=False,
            description="Generation timestamp (UTC)",
        ),
        ColumnSpec(
            "source", "string", nullable=False, description="Source tag (SYNTHETIC_V1)"
        ),
        ColumnSpec(
            "seed", "int64", nullable=False, description="Seed used for reproducibility"
        ),
    ],
)

DIM_EMPLOYEE_SCHEMA

TableSchema(name='dim_employee', columns=[ColumnSpec(name='employee_id', dtype='string', nullable=False, unique=True, description='Employee ID (e.g. E1000000)'), ColumnSpec(name='first_name', dtype='string', nullable=False, unique=False, description='Employee first name'), ColumnSpec(name='last_name', dtype='string', nullable=False, unique=False, description='Employee last name'), ColumnSpec(name='full_name', dtype='string', nullable=False, unique=False, description='Employee full name'), ColumnSpec(name='email', dtype='string', nullable=False, unique=True, description='Corporate email'), ColumnSpec(name='phone', dtype='string', nullable=True, unique=False, description='Phone number'), ColumnSpec(name='hire_date', dtype='datetime64[ns]', nullable=False, unique=False, description='Hiring date'), ColumnSpec(name='terminatation_date', dtype='datetime64[ns]', nullable=False, unique=False, description='Termination date (NaT if active)'), ColumnSpec(name='employment_status', dtype='string', 

In [6]:
# Summary table of the employee schema
schema_summary = pd.DataFrame(
    {
        "column": [col.name for col in DIM_EMPLOYEE_SCHEMA.columns],
        "dtype": [col.dtype for col in DIM_EMPLOYEE_SCHEMA.columns],
        "nullable": [col.nullable for col in DIM_EMPLOYEE_SCHEMA.columns],
        "unique": [col.unique for col in DIM_EMPLOYEE_SCHEMA.columns],
        "description": [col.description for col in DIM_EMPLOYEE_SCHEMA.columns],
    }
)

schema_summary

,column,dtype,nullable,unique,description
0,employee_id,string,False,True,Employee ID (e.g. E1000000)
1,first_name,string,False,False,Employee first name
2,last_name,string,False,False,Employee last name
3,full_name,string,False,False,Employee full name
4,email,string,False,True,Corporate email
5,phone,string,True,False,Phone number
6,hire_date,datetime64[ns],False,False,Hiring date
7,terminatation_date,datetime64[ns],False,False,Termination date (NaT if active)
8,employment_status,string,False,False,Active or Terminated
9,job_family,string,False,False,"ENGINEERING, SALES, PRODUCT, or G&A"


## Generator Interface

In [7]:
@dataclass
class BaseGenerator(ABC):
    @abstractmethod
    def generate(self) -> pd.DataFrame:
        """Generate a dataset."""
        raise NotImplementedError

    @abstractmethod
    def validate(self, df: pd.DataFrame) -> List[str]:
        """Return validation errors (empty if valid)"""
        raise NotImplementedError

    def profile(self, df: pd.DataFrame) -> Dict[str, Any]:
        """Basic dataset profiling."""
        return {
            "row_count": int(len(df)),
            "null_counts": df.isna().sum().to_dict(),
        }

## Data Generation
- Generate hire dates with growth from starting employee number to ending employee number over a specified date range.
- Create identity fields using `faker`
- Validate constraints and profile outputs


### Hire Date Generator
- Function to generate hire dates based on employee number growth over time
    - Assume exponential growth model for employee hires

In [8]:
def generate_hire_dates(
    company_start_date: pd.Timestamp,
    as_of_date: pd.Timestamp,
    initial_headcount: int,
    final_headcount: int,
    rng: np.random.Generator,
    growth_curve: str = "logistic",
    growth_steepness: float = 7.5,
) -> pd.DatetimeIndex:
    """
    Generate employee hire dates that model company headcount growth over time.

    The function enforces:
    - `initial_headcount` hires at (or effectively at) company start
    - total headcount reaching `final_headcount` by `as_of_date`
    - a configurable growth shape (linear or backloaded/logistic)

    Parameters
    ----------
    company_start_date : pd.Timestamp
        Date the company was founded / began hiring.
    as_of_date : pd.Timestamp
        Snapshot date by which final_headcount must be reached.
    initial_headcount : int
        Number of employees at company_start_date.
    final_headcount : int
        Total number of employees as of as_of_date.
    rng : np.random.Generator
        Seeded random number generator for reproducibility.
    growth_curve : str
        "linear" for uniform hiring, "logistic" for startup-style backloaded growth.
    growth_steepness : float
        Controls how aggressively hiring is backloaded when using logistic growth.

    Returns
    -------
    pd.DatetimeIndex
        Sorted hire dates of length `final_headcount`.
    """
    # Basic validation
    if final_headcount < initial_headcount:
        raise ValueError("final_headcount must be >= initial_headcount")
    if company_start_date >= as_of_date:
        raise ValueError("company_start_date must be < as_of_date")

    # Seed initial hires at the company founding
    remaining_hires = final_headcount - initial_headcount
    founding_hires = [company_start_date] * initial_headcount

    if remaining_hires == 0:
        return pd.DatetimeIndex(founding_hires)

    # Hiring window length (in days)
    hiring_window_days = int((as_of_date - company_start_date).days)
    if hiring_window_days <= 0:
        return pd.DatetimeIndex(founding_hires)

    # Growth curve handling - linearly spaced dates vs. exponential
    if growth_curve == "linear":
        # Uniform hiring time dates
        uniform_samples = rng.random(remaining_hires)
        day_offsets = (uniform_samples * hiring_window_days).astype(int)
    elif growth_curve == "logistic":
        uniform_samples = rng.random(remaining_hires)
        uniform_samples = np.clip(uniform_samples, 1e-9, 1 - 1e-9)
        logit_space = np.log(uniform_samples / (1 - uniform_samples))
        scaled_curve = 1 / (1 + np.exp(-logit_space / growth_steepness))
        day_offsets = (scaled_curve * hiring_window_days).astype(int)
    else:
        raise ValueError("growth_curve must be 'linear' or 'logistic'")

    # Add a small amount of noise to prevent any clustering
    jitter = rng.integers(low=0, high=3, size=remaining_hires)
    day_offsets = np.clip(day_offsets + jitter, 0, hiring_window_days)

    # Convert offsets to timestamps and return the starting member dates and new hires over time
    growth_hires = [
        company_start_date + pd.Timedelta(days=int(day)) for day in day_offsets
    ]
    all_hire_dates = pd.to_datetime(founding_hires + growth_hires).sort_values()
    return pd.DatetimeIndex(all_hire_dates)


# Validate
hire_dates = generate_hire_dates(
    company_start_date=pd.Timestamp(CONFIG.time.start_date),
    as_of_date=pd.Timestamp(CONFIG.time.as_of_date),
    initial_headcount=CONFIG.population.start_headcount,
    final_headcount=CONFIG.population.target_headcount,
    rng=rng,
    growth_curve="logistic",
    growth_steepness=3,
)

hire_dates[-5:]

DatetimeIndex(['2024-03-13', '2024-06-22', '2024-07-15', '2024-07-24',
               '2024-07-30'],
              dtype='datetime64[ns]', freq=None)

### Employee ID's

In [9]:
# ID number generator
def make_ids(prefix: str, width: int, size: int) -> List[str]:
    """Generate padded sequential ID's with prefix e.g. E100000 for employees"""
    return [f"{prefix}{i:0{width}d}" for i in range(1, size + 1)]

### Email
- Lowercase, space removed
- Function to handle duplicate names and generate appropriate email addresses

In [10]:
def clean_email_token(value: str) -> str:
    """Return a lowercase alphanumeric token suitable for emails."""
    value = value.lower().strip()
    return "".join(char for char in value if char.isalnum())

In [11]:
def build_emails(
    first_names: List[str],
    last_names: List[str],
    domain: str
) -> List[str]:
    """Create emails first.last@domain and append 2 or 3 for duplicates before the domain"""
    email_counts = {}
    emails = []

    for first, last in zip(first_names, last_names):
        concat_name = f"{clean_email_token(first)}.{clean_email_token(last)}"
        email_counts[concat_name] = email_counts.get(concat_name, 0) + 1
        count = email_counts[concat_name]
        email = f"{concat_name}{count}@{domain}" if count > 1 else f"{concat_name}@{domain}"
        emails.append(email)
    return emails

### Employee Record Generator
- Function to generate complete employee records using the defined schema and hire dates

In [12]:
@dataclass
class EmployeeGenerator(BaseGenerator):
    cfg: GlobalConfig

    def __post_init__(self) -> None:
        self._rng = _rng(self.cfg.run.seed)
        self._faker = Faker()
        self._faker.seed_instance(self.cfg.run.seed)

    def generate(self) -> pd.DataFrame:
        # Inputs
        employee_count = int(self.cfg.population.target_headcount)
        founding_headcount = int(self.cfg.population.start_headcount)

        # Generate employee ID's
        employee_ids = make_ids(
            prefix=self.cfg.ids.employee_id_prefix,
            width=self.cfg.ids.employee_id_width,
            size=employee_count,
        )

        # Generate hire dates
        hire_dates = generate_hire_dates(
            company_start_date=pd.Timestamp(self.cfg.time.start_date),
            as_of_date=pd.Timestamp(self.cfg.time.as_of_date),
            initial_headcount=founding_headcount,
            final_headcount=employee_count,
            rng=self._rng,
            growth_curve="logistic",
            growth_steepness=5,
        )

        # Job family
        job_families = sample_weighted(
            rng=self._rng,
            weight_by_key=self.cfg.dist.job_family_weights,
            sample_size=employee_count,
        )

        # NOTE: Update later
        departments = job_families

        # Locations
        locations = sample_weighted(
            rng=self._rng,
            weight_by_key=self.cfg.dist.location_weights,
            sample_size=employee_count,
        )

        # Job Levels
        levels: List[str] = []
        for family in job_families:
            job_level = sample_weighted(
                rng=self._rng,
                weight_by_key=self.cfg.dist.level_weights_by_job_family[family],
                sample_size=1,
            )[0]
            levels.append(job_level)
        
        # Other metadata per employee level
        first_names: List[str] = []
        last_names: List[str] = []
        phones: List[str] = []
        emails: List[str] = []

        domain = self.cfg.company.company_domain
        for employee_id in employee_ids:
            first_name = self._faker.first_name()
            last_name = self._faker.last_name()
            first_names.append(first_name)
            last_names.append(last_name)
            phones.append(self._faker.phone_number())
        
        full_names = [f"{first} {last}" for first, last in zip(first_names, last_names)]
        emails = build_emails(
            first_names=first_names,
            last_names=last_names,
            domain=self.cfg.company.company_domain
        )

        df = pd.DataFrame(
            {
                "employee_id": employee_ids,
                "first_name": first_names,
                "last_name": last_names,
                "full_name": full_names,
                "email": emails,
                "phone": phones,
                "hire_date": pd.to_datetime(hire_dates),
                "location": locations,
                "department": departments,
                "job_family": job_families,
                "level": levels,
            }
        )

        return df.sort_values(["hire_date", "employee_id"])

    def validate(self, df: pd.DataFrame) -> List[str]:
        errors: List[str] = []

        # Basic validation
        required_columns = {
            "employee_id",
            "full_name",
            "last_name",
            "full_name",
            "email",
            "phone",
            "hire_date",
            "job_family",
            "location",
            "level",
            "department",
        }
        missing_columns = required_columns - set(df.columns)
        if missing_columns:
            errors.append(f"Missing required columns: {sorted(missing_columns)}")

        if "employee_id" in df.columns:
            if df["employee_id"].isna().any():
                errors.append("Null employee_id values found")
            if df["employee_id"].duplicated().any():
                errors.append("Duplicate employee_id values found")

        if "hire_date" in df.columns:
            if df["hire_date"].isna().any():
                errors.append("Null hire_date values found")

        for col in ["location", "department", "job_family", "level"]:
            if col in df.columns and df[col].isna().any():
                errors.append(f"Null {col} values found")

        expected_rows = int(self.cfg.population.target_headcount)
        if len(df) != expected_rows:
            errors.append(f"Row count {len(df)} != target_headcount {expected_rows}")

        return errors


# Validate
gen = EmployeeGenerator(cfg=CONFIG)
employee = gen.generate()

errors = gen.validate(employee)
profile = gen.profile(employee)

display(errors, profile)
display(employee)

[]

{'row_count': 105,
 'null_counts': {'employee_id': 0,
  'first_name': 0,
  'last_name': 0,
  'full_name': 0,
  'email': 0,
  'phone': 0,
  'hire_date': 0,
  'location': 0,
  'department': 0,
  'job_family': 0,
  'level': 0}}

,employee_id,first_name,last_name,full_name,email,phone,hire_date,location,department,job_family,level
0,EMP000001,Danielle,Johnson,Danielle Johnson,danielle.johnson@gammasoftware.com,533-521-8196x001,2020-01-01,NY,SALES,SALES,L1
1,EMP000002,William,Johnson,William Johnson,william.johnson@gammasoftware.com,886.737.9402,2020-01-01,SF,ENGINEERING,ENGINEERING,L4
2,EMP000003,Joshua,Lewis,Joshua Lewis,joshua.lewis@gammasoftware.com,001-851-316-1559x40781,2020-01-01,NY,PRODUCT,PRODUCT,L3
3,EMP000004,Andrea,Calderon,Andrea Calderon,andrea.calderon@gammasoftware.com,+1-659-931-0341x316,2021-08-20,REMOTE,ENGINEERING,ENGINEERING,L4
4,EMP000005,Cynthia,Diaz,Cynthia Diaz,cynthia.diaz@gammasoftware.com,(653)541-9283x276,2021-12-04,NY,PRODUCT,PRODUCT,L4
...,...,...,...,...,...,...,...,...,...,...,...
100,EMP000101,Jennifer,Williams,Jennifer Williams,jennifer.williams@gammasoftware.com,(737)994-7383x473,2023-09-30,SF,ENGINEERING,ENGINEERING,L3
101,EMP000102,Steven,Campbell,Steven Campbell,steven.campbell@gammasoftware.com,(388)362-3924x0758,2023-12-11,SF,ENGINEERING,ENGINEERING,L4
102,EMP000103,Justin,Hines,Justin Hines,justin.hines@gammasoftware.com,878-726-1375x060,2023-12-27,REMOTE,PRODUCT,PRODUCT,L4
103,EMP000104,Joe,Vang,Joe Vang,joe.vang@gammasoftware.com,(930)951-5220x472,2024-01-06,NY,G&A,G&A,L3
